<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_02_data_and_sampling_distributions/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 2 folder
%cd chapter_02_data_and_sampling_distributions

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 156 (delta 93), reused 71 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 546.04 KiB | 9.93 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_02_data_and_sampling_distributions



# Chapter 02 Experiments: Data and Sampling Distributions

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 2: Data and Sampling Distributions

## Purpose

This notebook is for:
- Experimentation
- Intuition building
- Testing assumptions
- Learning through simulation
- Connecting theory to practice

Unlike `exercises.ipynb`, there are no fixed answers.

Goal:

> "What happens if I change the sampling process?"

---

## 🧪 How to Use This Notebook

- **Experiments ≠ Exercises**: These are open-ended explorations. There are no "right" answers.
- **Modify freely**: Change parameters, swap datasets, break things, and observe.
- **Document insights**: Add your observations in markdown cells as you go.
- **Save interesting outputs**: Export plots to `output/` for your portfolio.

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats
    from scipy.stats import sem, skew, kurtosis, norm, t, expon, uniform, shapiro
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

# Create output directory
os.makedirs('output', exist_ok=True)

print("Experiment notebook ready")
```
</details>



---

## Load Dataset

<details>
<summary>Click to view solution</summary>

```python
# Load state dataset
data_path = '../datasets/raw/state.csv'
if os.path.exists(data_path):
    state = pd.read_csv(data_path)
else:
    url = 'https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/state.csv'
    state = pd.read_csv(url)

state.head()
```
</details>

---

## Helper Functions

<details>
<summary>Click to view solution</summary>

```python
# Reusable bootstrap function
def bootstrap_statistic(data, statistic_func, R=1000, random_state=None):
    """Generic bootstrap function."""
    if random_state is not None:
        np.random.seed(random_state)
    
    bootstrap_stats = []
    n = len(data)
    
    for _ in range(R):
        resample = np.random.choice(data, size=n, replace=True)
        bootstrap_stats.append(statistic_func(resample))
    
    return np.array(bootstrap_stats)

# Generate population function
def generate_population(dist_type, n=100000, **params):
    """Generate populations with different distributions"""
    if dist_type == 'normal':
        return np.random.normal(**params, size=n)
    elif dist_type == 'exponential':
        return np.random.exponential(**params, size=n)
    elif dist_type == 'uniform':
        return np.random.uniform(**params, size=n)
    elif dist_type == 'binomial':
        return np.random.binomial(**params, size=n)
    elif dist_type == 'skewed_normal':
        base = np.random.normal(size=n)
        return np.where(base > 0, base * 2, base)
    elif dist_type == 'mixed':
        base = np.random.normal(100, 15, int(n * 0.95))
        outliers = np.random.choice([50, 200], size=int(n * 0.05))
        return np.concatenate([base, outliers])
    else:
        raise ValueError(f"Unknown distribution: {dist_type}")

# Sampling distribution function
def sampling_distribution(population, sample_size, n_samples=1000, statistic=np.mean):
    """Generate sampling distribution of a statistic"""
    stats_list = []
    for _ in range(n_samples):
        sample = np.random.choice(population, size=sample_size, replace=False)
        stats_list.append(statistic(sample))
    return np.array(stats_list)

# Bootstrap CI function
def bootstrap_ci(data, statistic=np.mean, confidence=0.95, R=2000, random_state=None):
    """Generic bootstrap confidence interval"""
    if random_state is not None:
        np.random.seed(random_state)
    
    bootstrap_stats = []
    n = len(data)
    
    for _ in range(R):
        resample = np.random.choice(data, size=n, replace=True)
        bootstrap_stats.append(statistic(resample))
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_stats, 100 * alpha/2)
    upper = np.percentile(bootstrap_stats, 100 * (1 - alpha/2))
    
    return {
        'estimate': statistic(data),
        'se_bootstrap': np.std(bootstrap_stats),
        'ci_lower': lower,
        'ci_upper': upper,
        'width': upper - lower
    }

def formula_ci_mean(data, confidence=0.95):
    """Formula-based CI for mean using t-distribution"""
    n = len(data)
    mean = np.mean(data)
    se = np.std(data, ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf((1 + confidence) / 2, df=n-1)
    
    return {
        'estimate': mean,
        'se_formula': se,
        'ci_lower': mean - t_crit * se,
        'ci_upper': mean + t_crit * se,
        'width': 2 * t_crit * se
    }

print("Helper functions loaded")
```
</details>


---

# Experiment 1: Simulating the Literary Digest Disaster

### Goal
Recreate the 1936 Literary Digest polling failure to understand how sample bias overwhelms sample size.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# True population: 60% Roosevelt, 40% Landon
def create_population(n=100000, p_roosevelt=0.60):
    """Create synthetic voting population"""
    return np.random.choice(['Roosevelt', 'Landon'], size=n, p=[p_roosevelt, 1-p_roosevelt])

# Biased sampling: affluent voters favor Landon
def biased_sample(population, affluent_frac=0.30, affluent_roosevelt_p=0.40, n_sample=2000000):
    """Simulate Literary Digest's biased sampling"""
    is_affluent = np.random.random(len(population)) < affluent_frac
    
    affluent_prefs = np.random.choice(['Roosevelt', 'Landon'],
                                       size=is_affluent.sum(),
                                       p=[affluent_roosevelt_p, 1-affluent_roosevelt_p])
    non_affluent_prefs = population[~is_affluent][:len(population) - is_affluent.sum()]
    
    biased_pool = np.concatenate([affluent_prefs, non_affluent_prefs])
    return np.random.choice(biased_pool, size=n_sample, replace=False)

# Random sampling (Gallup-style)
def random_sample(population, n=2000):
    """Simple random sample"""
    return np.random.choice(population, size=n, replace=False)

# Create population and draw samples
population = create_population()
biased = biased_sample(population)
random_samp = random_sample(population)

# Calculate results
def calculate_results(sample, name):
    roosevelt_pct = np.mean(sample == 'Roosevelt') * 100
    landon_pct = np.mean(sample == 'Landon') * 100
    return {'sample': name, 'Roosevelt': roosevelt_pct, 'Landon': landon_pct, 'n': len(sample)}

results = pd.DataFrame([
    calculate_results(population, 'True Population'),
    calculate_results(biased, 'Biased Sample (Literary Digest)'),
    calculate_results(random_samp, 'Random Sample (Gallup)')
])

print(results.to_string(index=False))
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Visualize the bias
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
categories = ['Roosevelt', 'Landon']
x = np.arange(len(categories))
colors = ['skyblue', 'salmon', 'lightgreen']

for idx, (_, row) in enumerate(results.iterrows()):
    values = [row['Roosevelt']/100, row['Landon']/100]
    axes[idx].bar(x, values, color=colors[idx], edgecolor='black')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(categories)
    axes[idx].set_ylabel('Proportion')
    axes[idx].set_title(f'{row["sample"]}\n(n={row["n"]:,})')
    axes[idx].set_ylim(0, 1)
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Sample Bias: Size ≠ Quality', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Calculate prediction error
true_roosevelt = results.iloc[0]['Roosevelt']
for _, row in results.iloc[1:].iterrows():
    error = abs(row['Roosevelt'] - true_roosevelt)
    print(f"{row['sample']}: Error = {error:.1f} percentage points")
```
</details>

### 🤔 Reflection Prompts
1. How much bias is "acceptable" for your use case?
2. If you could only afford n=2,000, would you prefer a biased large sample or random small sample?
3. **ML Relevance**: How might similar biases appear in training data for ML models (e.g., user behavior data)?
4. Try varying `affluent_roosevelt_p` — at what point does the biased sample become "good enough"?



---

# Experiment 2: How Much Do Samples Vary?

## Question

If we repeatedly sample data, do results stay the same?

### Hypothesis

Different samples should produce slightly different means.

<details>
<summary>Click to view solution</summary>

```python
sample_means = []

for _ in range(100):
    sample = state.sample(n=10, replace=True)
    sample_means.append(sample["Population"].mean())

plt.figure(figsize=(10, 6))
sns.histplot(sample_means, bins=20, kde=True)
plt.axvline(state["Population"].mean(), linestyle="--", label="Population Mean")
plt.legend()
plt.title("How Sample Means Vary")
plt.show()
```
</details>

## Reflection

Questions:
1. Why are means different?
2. Are they wildly different?
3. Why does this matter in real life?



---

# Experiment 3: Sample Size vs Stability

## Question

What happens if sample size increases?

### Hypothesis

Larger samples should produce more stable estimates.

<details>
<summary>Click to view solution</summary>

```python
sizes = [5, 10, 30, 50]
results = {}

for n in sizes:
    means = []
    for _ in range(1000):
        sample = state.sample(n=n, replace=True)
        means.append(sample["Population"].mean())
    results[n] = means

fig, ax = plt.subplots(2, 2, figsize=(14, 10))
ax = ax.flatten()

for i, n in enumerate(sizes):
    sns.histplot(results[n], bins=25, kde=True, ax=ax[i])
    ax[i].set_title(f"Sample Size = {n}")

plt.tight_layout()
plt.show()
```
</details>

## Reflection

Questions:
1. Which sample size is most stable?
2. Why?
3. What does this teach about ML datasets?



---

# Experiment 4: Biased Sampling

## Question

Can bias destroy analysis?

### Hypothesis

Bad sampling creates misleading conclusions.

<details>
<summary>Click to view solution</summary>

```python
random_sample = state.sample(n=20, random_state=42)
biased_sample = state.nlargest(20, "Population")

comparison = pd.DataFrame({
    "Mean Population": [
        state["Population"].mean(),
        random_sample["Population"].mean(),
        biased_sample["Population"].mean()
    ]
}, index=["Population", "Random Sample", "Biased Sample"])

print(comparison)

comparison.plot(kind="bar", figsize=(8, 5))
plt.title("Bias in Sampling")
plt.ylabel("Population")
plt.show()
```
</details>

## Reflection

Questions:
1. Why is biased sampling dangerous?
2. Can big datasets still be biased?
3. What ML problems come from bias?



---

# Experiment 5: Regression to the Mean — How Strong Is the Effect?

### Goal
Quantify how regression to the mean varies with noise level and selection threshold.

<details>
<summary>Click to view solution</summary>

```python
def simulate_regression_to_mean(n=1000, skill_sd=1, noise_sd=2, top_pct=10, seed=None):
    """Simulate performance = skill + noise across two periods."""
    if seed is not None:
        np.random.seed(seed)
    
    skill = np.random.normal(0, skill_sd, n)
    noise_p1 = np.random.normal(0, noise_sd, n)
    noise_p2 = np.random.normal(0, noise_sd, n)
    
    perf_p1 = skill + noise_p1
    perf_p2 = skill + noise_p2
    
    threshold = np.percentile(perf_p1, 100 - top_pct)
    top_mask = perf_p1 >= threshold
    
    selected_p1 = perf_p1[top_mask]
    selected_p2 = perf_p2[top_mask]
    
    return {
        'n_selected': top_mask.sum(),
        'p1_mean': selected_p1.mean(),
        'p2_mean': selected_p2.mean(),
        'regression_amount': selected_p1.mean() - selected_p2.mean(),
        'p1_sd': selected_p1.std(),
        'p2_sd': selected_p2.std(),
        'correlation': np.corrcoef(selected_p1, selected_p2)[0, 1],
        'stability_rate': np.mean(perf_p2[top_mask] >= np.percentile(perf_p2, 100 - top_pct))
    }

# Vary noise-to-skill ratio
noise_levels = [0.5, 1, 2, 4, 8]
results = []

for noise_sd in noise_levels:
    metrics = simulate_regression_to_mean(noise_sd=noise_sd, seed=42)
    metrics['noise_sd'] = noise_sd
    results.append(metrics)

results_df = pd.DataFrame(results)

# Plot regression amount vs. noise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(results_df['noise_sd'], results_df['regression_amount'], 'o-', linewidth=2)
axes[0].set_xlabel('Noise Standard Deviation')
axes[0].set_ylabel('Regression Amount (P1 Mean - P2 Mean)')
axes[0].set_title('More Noise → More Regression')
axes[0].grid(alpha=0.3)

axes[1].plot(results_df['noise_sd'], results_df['correlation'], 's--', color='orange', linewidth=2)
axes[1].set_xlabel('Noise Standard Deviation')
axes[1].set_ylabel('Correlation (P1 vs P2 for Top Performers)')
axes[1].set_title('More Noise → Lower Correlation')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(results_df[['noise_sd', 'regression_amount', 'correlation', 'stability_rate']].to_string(index=False))
```
</details>

### 🤔 Reflection Prompts
1. At what noise level does regression become "problematic" for your application?
2. How would you design an experiment to minimize regression-to-mean artifacts?
3. **ML Relevance**: How might this affect A/B test interpretation when selecting "top" variants?
4. Try varying `top_pct` — does selecting the top 1% vs. top 25% change the regression amount?



---

# Experiment 6: Central Limit Theorem — How "Large" Is Large Enough?

### Goal
Test how sample size requirements for CLT vary with population distribution shape.

<details>
<summary>Click to view solution</summary>

```python
def assess_normality(sample, alpha=0.05):
    """Quick normality assessment using skewness and Shapiro-Wilk"""
    sk = skew(sample)
    if len(sample) > 5000:
        sample = sample[:5000]
    _, p_value = shapiro(sample)
    return {
        'skewness': sk,
        'shapiro_p': p_value,
        'is_normal': p_value > alpha and abs(sk) < 0.5
    }

distributions = {
    'Normal': {'dist_type': 'normal', 'loc': 0, 'scale': 1},
    'Exponential': {'dist_type': 'exponential', 'scale': 1},
    'Uniform': {'dist_type': 'uniform', 'low': 0, 'high': 10},
    'Binomial(n=10,p=0.3)': {'dist_type': 'binomial', 'n': 10, 'p': 0.3},
    'Skewed Normal': {'dist_type': 'skewed_normal'}
}

sample_sizes = [5, 10, 25, 50, 100]
results_clt = []

for dist_name, dist_params in distributions.items():
    population = generate_population(**dist_params)
    
    for n in sample_sizes:
        sample_means = sampling_distribution(population, sample_size=n)
        normality = assess_normality(sample_means)
        
        results_clt.append({
            'distribution': dist_name,
            'sample_size': n,
            'mean_of_means': sample_means.mean(),
            'se_empirical': sample_means.std(),
            'se_theoretical': population.std() / np.sqrt(n),
            'skewness': normality['skewness'],
            'shapiro_p': normality['shapiro_p'],
            'is_normal': normality['is_normal']
        })

results_clt_df = pd.DataFrame(results_clt)
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Plot sampling distributions for Exponential distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

dist_to_plot = 'Exponential'
population = generate_population(**distributions[dist_to_plot])

for idx, n in enumerate(sample_sizes[:6]):
    sample_means = sampling_distribution(population, sample_size=n)
    
    axes[idx].hist(sample_means, bins=30, density=True, alpha=0.7, edgecolor='black')
    
    mu, sigma = sample_means.mean(), sample_means.std()
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
    axes[idx].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2)
    
    axes[idx].axvline(population.mean(), color='green', linestyle='--', label='Pop Mean')
    axes[idx].set_title(f'n={n}\nSkew: {skew(sample_means):.2f}')
    axes[idx].set_xlabel('Sample Mean')
    axes[idx].set_ylabel('Density')
    axes[idx].grid(alpha=0.3)
    if idx == 0:
        axes[idx].legend(fontsize=8)

plt.suptitle(f'CLT Convergence: {dist_to_plot} Population', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print(f"\nNormality Assessment (Shapiro-Wilk p > 0.05 and |skew| < 0.5):")
pivot = results_clt_df.pivot(index='sample_size', columns='distribution', values='is_normal')
print(pivot.to_string())
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Original CLT experiment with state data
skewed_data = np.random.exponential(scale=2, size=10000)

plt.figure(figsize=(10, 6))
sns.histplot(skewed_data, bins=50, kde=True)
plt.title("Original Skewed Distribution")
plt.show()

sample_means_clt = []
for _ in range(1000):
    sample = np.random.choice(skewed_data, size=30, replace=True)
    sample_means_clt.append(sample.mean())

plt.figure(figsize=(10, 6))
sns.histplot(sample_means_clt, bins=30, kde=True)
plt.title("Sampling Distribution (CLT)")
plt.show()
```
</details>

### 🤔 Reflection Prompts
1. Which distributions require the largest n for CLT to "work"?
2. Is the theoretical SE formula accurate even when the sampling distribution isn't normal?
3. **ML Relevance**: When building confidence intervals for model metrics, should you worry about CLT assumptions?
4. Try using median instead of mean — does CLT still apply?



---

# Experiment 7: Standard Error vs Sample Size

## Question

What happens to standard error as sample size grows?

### Hypothesis

Standard error should decrease.

<details>
<summary>Click to view solution</summary>

```python
sample_sizes = np.arange(5, 100, 5)
errors = []

for n in sample_sizes:
    sample = state.sample(n=n, replace=True)
    errors.append(sem(sample["Population"]))

plt.figure(figsize=(10, 6))
plt.plot(sample_sizes, errors, marker="o")
plt.xlabel("Sample Size")
plt.ylabel("Standard Error")
plt.title("Sample Size vs Standard Error")
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Reflection

Questions:
1. Why does error decrease?
2. Why not just collect infinite data?
3. What trade-offs exist?



---

# Experiment 8: Standard Error — The Square Root of n Rule

### Goal
Verify that SE decreases as 1/√n and explore when this rule breaks down.

<details>
<summary>Click to view solution</summary>

```python
def estimate_se_empirical(population, sample_sizes, n_replications=500, statistic=np.mean):
    """Empirically estimate SE for different sample sizes"""
    results = []
    
    for n in sample_sizes:
        sample_stats = []
        for _ in range(n_replications):
            sample = np.random.choice(population, size=n, replace=False)
            sample_stats.append(statistic(sample))
        
        results.append({
            'n': n,
            'empirical_se': np.std(sample_stats),
            'theoretical_se': np.std(population) / np.sqrt(n) if statistic == np.mean else None
        })
    
    return pd.DataFrame(results)

# Create populations with different characteristics
populations_se = {
    'Normal': generate_population('normal', n=50000, loc=100, scale=15),
    'Skewed': generate_population('exponential', n=50000, scale=10),
    'With Outliers': np.concatenate([
        np.random.normal(100, 15, 47500),
        np.random.choice([10, 200], size=2500)
    ])
}

sample_sizes_se = [10, 25, 50, 100, 200, 400]
all_results_se = []

for pop_name, population in populations_se.items():
    se_results = estimate_se_empirical(population, sample_sizes_se)
    se_results['population'] = pop_name
    all_results_se.append(se_results)

all_df_se = pd.concat(all_results_se, ignore_index=True)

# Plot SE vs. sample size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-log plot
for pop_name in populations_se.keys():
    subset = all_df_se[all_df_se['population'] == pop_name]
    axes[0].loglog(subset['n'], subset['empirical_se'], 'o-', label=pop_name, markersize=6)
    
    if pop_name == 'Normal':
        theoretical = np.std(populations_se['Normal']) / np.sqrt(np.array(sample_sizes_se))
        axes[0].loglog(sample_sizes_se, theoretical, 'k--', label='Theoretical: sigma/sqrt(n)')

axes[0].set_xlabel('Sample Size (n)')
axes[0].set_ylabel('Standard Error')
axes[0].set_title('SE vs. Sample Size (Log-Log)')
axes[0].legend()
axes[0].grid(alpha=0.3, which='both')

# Ratio of empirical to theoretical
normal_subset = all_df_se[all_df_se['population'] == 'Normal']
axes[1].plot(normal_subset['n'], normal_subset['empirical_se'] / normal_subset['theoretical_se'],
             'o-', markersize=6)
axes[1].axhline(y=1, color='green', linestyle='--', label='Perfect Match')
axes[1].set_xlabel('Sample Size (n)')
axes[1].set_ylabel('Empirical SE / Theoretical SE')
axes[1].set_title('How Well Does Theory Match Practice?')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nEmpirical SE / Theoretical SE Ratios (Normal Population):")
print(normal_subset[['n', 'empirical_se', 'theoretical_se']].to_string(index=False))
```
</details>

### 🤔 Reflection Prompts
1. Does the 1/√n rule hold for skewed populations? For statistics other than the mean?
2. How large does n need to be before empirical SE matches theory within 10%?
3. **ML Relevance**: How does this inform decisions about train/validation split sizes?
4. What happens if you sample *with* replacement instead of without?



---

# Experiment 9: Bootstrap Stability

## Question

Does bootstrap produce stable estimates?

<details>
<summary>Click to view solution</summary>

```python
bootstrap_means = []

for _ in range(1000):
    sample = state.sample(frac=1, replace=True)
    bootstrap_means.append(sample["Population"].mean())

plt.figure(figsize=(10, 6))
sns.histplot(bootstrap_means, bins=30, kde=True)
plt.title("Bootstrap Mean Stability")
plt.show()

print("Bootstrap Mean:", np.mean(bootstrap_means))
print("Bootstrap Std:", np.std(bootstrap_means))
print("Original Mean:", state["Population"].mean())
```
</details>

## Reflection

Questions:
1. Why does bootstrap work?
2. Why sample with replacement?
3. How is bootstrap used in ML?



---

# Experiment 10: Bootstrap vs. Formula — When Do They Disagree?

### Goal
Compare bootstrap and formula-based confidence intervals under different conditions.

<details>
<summary>Click to view solution</summary>

```python
conditions = [
    {'name': 'Normal, n=30', 'dist': 'normal', 'n': 30, 'params': {'loc': 100, 'scale': 15}},
    {'name': 'Normal, n=100', 'dist': 'normal', 'n': 100, 'params': {'loc': 100, 'scale': 15}},
    {'name': 'Exponential, n=30', 'dist': 'exponential', 'n': 30, 'params': {'scale': 10}},
    {'name': 'Exponential, n=100', 'dist': 'exponential', 'n': 100, 'params': {'scale': 10}},
    {'name': 'With Outliers', 'dist': 'mixed', 'n': 100, 'params': {}},
]

comparison_results = []

for cond in conditions:
    if cond['dist'] == 'mixed':
        data = generate_population('mixed', cond['n'])
    else:
        data = generate_population(cond['dist'], cond['n'], **cond['params'])
    
    boot = bootstrap_ci(data, np.mean, R=2000, random_state=42)
    formula = formula_ci_mean(data)
    
    comparison_results.append({
        'condition': cond['name'],
        'true_mean': data.mean(),
        'boot_width': boot['width'],
        'formula_width': formula['width'],
        'width_ratio': boot['width'] / formula['width'],
        'overlap': not (boot['ci_upper'] < formula['ci_lower'] or formula['ci_upper'] < boot['ci_lower']),
    })

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Visualize CI comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Width comparison
axes[0].barh(comparison_df['condition'], comparison_df['boot_width'],
             alpha=0.7, label='Bootstrap', color='skyblue')
axes[0].barh(comparison_df['condition'], comparison_df['formula_width'],
             alpha=0.5, label='Formula', color='salmon')
axes[0].set_xlabel('CI Width')
axes[0].set_title('CI Width: Bootstrap vs. Formula')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Coverage simulation for Exponential population
np.random.seed(42)
data_exp = generate_population('exponential', n=100, scale=10)
true_mean_exp = data_exp.mean()

n_sim = 200
boot_covers = 0
formula_covers = 0

for i in range(n_sim):
    sample = np.random.choice(data_exp, size=50, replace=False)
    
    boot = bootstrap_ci(sample, np.mean, R=500)
    formula = formula_ci_mean(sample)
    
    if boot['ci_lower'] <= true_mean_exp <= boot['ci_upper']:
        boot_covers += 1
    if formula['ci_lower'] <= true_mean_exp <= formula['ci_upper']:
        formula_covers += 1

axes[1].bar(['Bootstrap', 'Formula'], [boot_covers/n_sim, formula_covers/n_sim],
            color=['skyblue', 'salmon'], edgecolor='black')
axes[1].axhline(y=0.95, color='green', linestyle='--', label='Target: 95%')
axes[1].set_ylabel('Empirical Coverage')
axes[1].set_title(f'Coverage Simulation (n_sim={n_sim})\nExponential Population, n=50')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
```
</details>

### 🤔 Reflection Prompts
1. When do bootstrap and formula CIs disagree most? What does this tell you?
2. Is wider always better for a confidence interval?
3. **ML Relevance**: Should you use bootstrap CIs for model performance metrics?
4. Try bootstrapping the median — why can't you use a formula for that?


---

# Experiment 11: Confidence Interval Width

## Question

What affects confidence interval width?

<details>
<summary>Click to view solution</summary>

```python
sample_sizes_ci = [10, 30, 50]

for n in sample_sizes_ci:
    means = []
    for _ in range(1000):
        sample = state.sample(n=n, replace=True)
        means.append(sample["Population"].mean())
    
    ci = np.percentile(means, [2.5, 97.5])
    
    print(f"Sample Size {n}")
    print(f"Confidence Interval: [{ci[0]:,.0f}, {ci[1]:,.0f}]")
    print(f"Width: {ci[1] - ci[0]:,.0f}")
    print()
```
</details>

## Reflection

Questions:
1. Which interval is smallest?
2. Why?
3. Why do businesses care about uncertainty?


---

# Experiment 12: QQ-Plots as a Diagnostic Tool

### Goal
Use QQ-plots to assess normality assumptions and understand their limitations.

<details>
<summary>Click to view solution</summary>

```python
def create_qq_comparison(distributions_dict, sample_size=200, seed=42):
    """Create QQ-plots comparing different distributions to normal"""
    np.random.seed(seed)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, (name, dist_func) in enumerate(distributions_dict.items()):
        data = dist_func(size=sample_size)
        
        axes[idx].hist(data, bins=25, density=True, alpha=0.6, edgecolor='black')
        x = np.linspace(data.min(), data.max(), 100)
        axes[idx].plot(x, stats.norm.pdf(x, data.mean(), data.std()), 'r-', linewidth=2)
        axes[idx].set_title(f'{name}\nSkew: {skew(data):.2f}, Kurt: {kurtosis(data):.2f}')
        axes[idx].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # QQ-plots
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, (name, dist_func) in enumerate(distributions_dict.items()):
        data = dist_func(size=sample_size)
        stats.probplot(data, dist="norm", plot=axes[idx])
        axes[idx].set_title(f'QQ-Plot: {name}')
        axes[idx].grid(alpha=0.3)
    
    plt.suptitle('QQ-Plots: Deviations from Normality', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

distributions_qq = {
    'Normal(0,1)': lambda size: np.random.normal(0, 1, size),
    'Exponential(1)': lambda size: np.random.exponential(1, size),
    't(df=3)': lambda size: stats.t.rvs(df=3, size=size),
    'Mixture': lambda size: np.concatenate([
        np.random.normal(-2, 1, size//2),
        np.random.normal(2, 1, size//2)
    ])
}

create_qq_comparison(distributions_qq)
```
</details>

<details>
<summary>Click to view solution</summary>

```python
def quantify_qq_deviation(data, n_points=100):
    """Quantify how far QQ-plot deviates from diagonal"""
    theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, n_points))
    sample_quantiles = np.percentile(data, np.linspace(1, 99, n_points))
    
    sample_quantiles_std = (sample_quantiles - sample_quantiles.mean()) / sample_quantiles.std()
    
    deviations = np.abs(sample_quantiles_std - theoretical_quantiles)
    
    return {
        'max_deviation': deviations.max(),
        'mean_deviation': deviations.mean(),
        'tail_deviation': deviations[-10:].mean()
    }

# Apply to different sample sizes
np.random.seed(42)
exp_data = np.random.exponential(1, 1000)

sample_sizes_qq = [50, 100, 200, 500]
deviations = []

for n in sample_sizes_qq:
    sample = exp_data[:n]
    dev = quantify_qq_deviation(sample)
    dev['n'] = n
    deviations.append(dev)

dev_df = pd.DataFrame(deviations)
print(dev_df.to_string(index=False))

# Plot deviation vs. sample size
plt.figure(figsize=(8, 5))
plt.plot(dev_df['n'], dev_df['mean_deviation'], 'o-', label='Mean Deviation')
plt.plot(dev_df['n'], dev_df['tail_deviation'], 's--', label='Tail Deviation')
plt.xlabel('Sample Size')
plt.ylabel('QQ-Plot Deviation from Normal')
plt.title('Does Larger n Make Non-Normal Data Look More Normal?')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

### 🤔 Reflection Prompts
1. Can you "see" non-normality better in a histogram or a QQ-plot?
2. Does increasing sample size make non-normal data appear more normal in QQ-plots?
3. **ML Relevance**: When should you worry about non-normal residuals in regression?
4. Try standardizing data before QQ-plot — does it help or hurt interpretation?


---

# Experiment 13: Your Own Hypothesis — Test It!

### Template for Personal Experiments

<details>
<summary>Click to view solution</summary>

```python
# ### My Hypothesis: [Describe your question]
# Example: "Does the bootstrap work better for the median than for the mean with skewed data?"

np.random.seed(42)

# Generate skewed population
population = np.random.exponential(scale=10, size=10000)

# Take a sample
sample = np.random.choice(population, size=50, replace=False)

# Bootstrap both mean and median
boot_mean = bootstrap_ci(sample, np.mean, R=2000, random_state=42)
boot_median = bootstrap_ci(sample, np.median, R=2000, random_state=42)

print(f"Mean CI: [{boot_mean['ci_lower']:.2f}, {boot_mean['ci_upper']:.2f}]")
print(f"Median CI: [{boot_median['ci_lower']:.2f}, {boot_median['ci_upper']:.2f}]")

# Compare widths
print(f"Mean CI width: {boot_mean['width']:.2f}")
print(f"Median CI width: {boot_median['width']:.2f}")

# ### My Insight:
# [Write your conclusion here]
```
</details>

### Prompt Ideas for Your Own Experiments
- How does the bootstrap perform with very small samples (n < 20)?
- What happens if I bootstrap a correlation coefficient vs. using Fisher's z-transform?
- Can I use the bootstrap to estimate the sampling distribution of R² in regression?
- How does stratified bootstrap sampling compare to simple bootstrap for imbalanced data?
- What if I bootstrap with different values of R (100 vs. 10,000) — when does it stabilize?



---

# ML Relevance

## Why Chapter 2 matters in ML

Machine learning trains on samples, not reality itself.

Bad sampling causes:
- Unfair models
- Biased systems
- Weak generalisation

Bootstrap powers:
- Random forests
- Bagging
- Model uncertainty estimation

Confidence intervals help estimate:
- Prediction uncertainty
- Metric reliability
- A/B test significance

Train/test split is fundamentally about sampling.

---

# Personal Observations

## What surprised me?

Write observations here.

---

## Questions I still have

Add doubts here.

---

## Biggest takeaway

Write your main learning from Chapter 2.

---

## 📊 Exporting Your Findings

<details>
<summary>Click to view solution</summary>

```python
# Save the CLT convergence plot
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

population = generate_population('exponential', n=100000, scale=1)
sample_means_clt = sampling_distribution(population, sample_size=50, n_samples=2000)

plt.figure(figsize=(8, 6))
plt.hist(sample_means_clt, bins=40, density=True, alpha=0.7, edgecolor='black', color='skyblue')

mu, sigma = sample_means_clt.mean(), sample_means_clt.std()
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2.5, label='Normal Approximation')

plt.axvline(population.mean(), color='green', linestyle='--', label=f'Population Mean: {population.mean():.2f}')
plt.xlabel('Sample Mean')
plt.ylabel('Density')
plt.title('Sampling Distribution of Mean\nExponential Population, n=50', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.4, linestyle='--')
plt.tight_layout()

plt.savefig(f'{output_dir}/clt_exponential_n50.png', dpi=300, bbox_inches='tight')
print(f"Saved plot to {output_dir}/clt_exponential_n50.png")
plt.show()
```
</details>



---

## 📝 Experiment Log

| Date | Experiment | Key Insight | ML Relevance |
|------|-----------|-------------|--------------|
| | Literary Digest simulation | Sample bias > sample size | Training data representativeness matters more than volume |
| | CLT convergence | Skewed populations need n>50 | Don't assume normality for small-sample metrics |
| | Bootstrap vs. formula | Bootstrap wider for skewed data | Use bootstrap for robust uncertainty estimates |
| | Regression to mean | More noise = more regression | Validate model improvements on fresh holdout data |
| | QQ-plot diagnostics | QQ-plots reveal tail deviations | Check residual normality for inference validity |

*Update this table as you complete experiments.*

---

## Progress Checklist

- [ ] Explored sampling variability
- [ ] Compared sample sizes
- [ ] Tested sampling bias
- [ ] Simulated Literary Digest disaster
- [ ] Quantified regression to the mean
- [ ] Verified CLT with multiple distributions
- [ ] Investigated standard error vs sample size
- [ ] Verified square root of n rule
- [ ] Practised bootstrap
- [ ] Compared bootstrap vs formula CIs
- [ ] Compared confidence intervals across sample sizes
- [ ] Created QQ-plot diagnostics
- [ ] Connected concepts to ML
- [ ] Created personal experiment
- [ ] Exported key plots

---

## ✅ Next Steps

1. **Pick 2-3 experiments** to run deeply this week.
2. **Modify parameters** and observe how results change.
3. **Export** your favorite plot to `output/` for your portfolio.

---

> **Pro Tip**: Keep an `experiments_log.md` in your `notes/` folder where you briefly document:  
> - What you tried  
> - What surprised you  
> - What you'll try next  
> This becomes invaluable during interview prep or when revisiting concepts months later.

*Happy experimenting! 🧪✨*